In [100]:
import torch
import torch.nn as nn
import numpy as np
import os
import matplotlib.pyplot as plt

In [101]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    gpu_info = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_info.name}")
    print(f"GPU memory: {gpu_info.total_memory / 1024**2:.2f} MB")

GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU memory: 8187.50 MB


In [102]:
class DeepONet_g(nn.Module):
    def __init__(self, g_dim, hidden_dim):
        super().__init__()
        # Branch Net dim_input = 1 + f_dim + g_dim
        self.g_dim = g_dim
        self.branch_net = nn.Sequential(
            nn.Linear(1 + g_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.trunk_net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, k, g, x):
        """
        k: (B, 1)
        g: (B, g_dim)
        x: (M, 2)   
        """
        branch_input = torch.cat([k, g], dim=1)  # shape (B, 1+f_dim+g_dim)
        branch_out = self.branch_net(branch_input)  # shape (B, H)
        trunk_out = self.trunk_net(x)               # shape (M, 2)

        return torch.einsum('bi,mi->bm', branch_out, trunk_out)

In [103]:
class DeepONet_f(nn.Module):
    def __init__(self, f_dim, hidden_dim):
        super().__init__()
        # Branch Net dim_input = 1 + f_dim + g_dim
        self.f_dim = f_dim
        self.branch_net = nn.Sequential(
            nn.Linear(1 + f_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.trunk_net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, k, f, x):
        """
        k: (B, 1)
        f: (B, f_dim)
        x: (M, 2)   
        """
        branch_input = torch.cat([k, f], dim=1)  # shape (B, 1+f_dim+g_dim)
        branch_out = self.branch_net(branch_input)  # shape (B, H)
        trunk_out = self.trunk_net(x)               # shape (M, 2)

        return torch.einsum('bi,mi->bm', branch_out, trunk_out)

In [104]:
class GreenFun(nn.Module):
    def __init__(self, N, N_quadrature):     
        super(GreenFun, self).__init__()
        self.N = N
        self.N_quad = N_quadrature
        self.tau_layer = nn.Sequential(nn.Linear(1, N_quadrature), nn.ReLU(), nn.Linear(N_quadrature, N_quadrature), nn.ReLU(), nn.Linear(N_quadrature, N_quadrature), nn.ReLU(), nn.Linear(N_quadrature, N_quadrature))
        self.G_layer = nn.Sequential(nn.Linear(2, N_quadrature), nn.ReLU(), nn.Linear(N_quadrature, N_quadrature), nn.ReLU(), nn.Linear(N_quadrature, N_quadrature), nn.ReLU(), nn.Linear(N_quadrature, N_quadrature))

    def forward(self, f, x, tau):    # f: (batch_size, N_quad), x: (N, 2), tau: (batch_size, 1)
        T = self.tau_layer(torch.sqrt(tau))
        G = self.G_layer(x)     # G is (N, N_quad) with G(i, j) = G((x_i, y_i); (x_quad_j, y_quad_j))
        output = torch.matmul(f * T, G.t()) 
        output = output / self.N_quad 
        return output       # output: (batch_size, N)

class MLP(torch.nn.Module):
    def __init__(self, n_input, n_hidden, n_output, n_layers):
        super(MLP, self).__init__()
        self.act = nn.ReLU()
        self.layin = torch.nn.Linear(n_input, n_hidden)
        self.hidden_layers = torch.nn.ModuleList([torch.nn.Linear(n_hidden, n_hidden) for _ in range(n_layers)])
        self.layout = torch.nn.Linear(n_hidden, n_output)
    def forward(self, g):
       g = self.layin(g)
       g = self.act(g)
       for layer in self.hidden_layers:
           g = layer(g)
           g = self.act(g)
       h = self.layout(g)
       return h
    
class DeepONet(torch.nn.Module):
    def __init__(self, n_g, n_h, n_hidden, n_layers):
        super(DeepONet, self).__init__()
        self.for_tau = MLP(1, n_g, n_hidden, n_layers)
        self.for_g_1 = torch.nn.Linear(n_g, n_hidden)
        self.for_g_2 = torch.nn.Linear(n_hidden, n_h)
    def forward(self, tau, g):      #  tau: scalar, g: (batch_size, n_g)
       tau = tau + torch.zeros_like(g[:, 1].view(-1, 1))
       tau = self.for_tau(tau)
       g = self.for_g_1(g)
       h = self.for_g_2(tau * g)
       return h

In [105]:
NN = 40
M = (NN + 1) * (NN + 1)

net_g = DeepONet_g(g_dim = 4 * NN, hidden_dim = M // 4).to(device)
net_g.load_state_dict(torch.load("./DeepONet_g/net.pth", map_location=device))
net_g.eval()  

net_f = DeepONet_f(f_dim = M, hidden_dim = M // 4).to(device)
net_f.load_state_dict(torch.load("./DeepONet_f/net.pth", map_location=device))
net_f.eval()  

DeepONet_f(
  (branch_net): Sequential(
    (0): Linear(in_features=1682, out_features=420, bias=True)
    (1): ReLU()
    (2): Linear(in_features=420, out_features=420, bias=True)
    (3): ReLU()
    (4): Linear(in_features=420, out_features=420, bias=True)
    (5): ReLU()
    (6): Linear(in_features=420, out_features=420, bias=True)
    (7): ReLU()
    (8): Linear(in_features=420, out_features=420, bias=True)
  )
  (trunk_net): Sequential(
    (0): Linear(in_features=2, out_features=420, bias=True)
    (1): ReLU()
    (2): Linear(in_features=420, out_features=420, bias=True)
    (3): ReLU()
    (4): Linear(in_features=420, out_features=420, bias=True)
    (5): ReLU()
    (6): Linear(in_features=420, out_features=420, bias=True)
    (7): ReLU()
    (8): Linear(in_features=420, out_features=420, bias=True)
  )
)

In [106]:
N = 41
x_pt = torch.zeros(N * N, 2).to(device)
for i in range(N):
    for j in range(N):
        x_pt[i * N + j, 0] = 1 * j / (N - 1) 
        x_pt[i * N + j, 1] = 1 * i / (N - 1)  

boundary_mask = (
    (x_pt[:, 0] == 0) | (x_pt[:, 0] == 1) |  
    (x_pt[:, 1] == 0) | (x_pt[:, 1] == 1)   
)
x_bd_pt = x_pt[boundary_mask] 
x_int_pt = x_pt[~boundary_mask]  

netF = GreenFun(N * N, N * N).to(device)
netF.load_state_dict(torch.load("../Heat&Wave/GreenNet1/net.pth", map_location = device))

Ng = 128
netg = DeepONet(Ng * 4, Ng * 4, Ng * 3, 2).to(device)
netg.load_state_dict(torch.load("../Heat&Wave/BINet+OL1/net.pth", map_location = device))

def generate_square_points(n):
    # Boundary points per side without including corners
    boundary_spacing = np.linspace(1 / (2 * n), 1 - 1 / (2 * n), n)
    
    # Bottom boundary (x varies, y is 0)
    bottom_boundary = np.column_stack((boundary_spacing, np.zeros(n)))
    # Right boundary (x is 1, y varies)
    right_boundary = np.column_stack((np.ones(n), boundary_spacing))
    # Top boundary (x varies, y is 1)
    top_boundary = np.column_stack((boundary_spacing[::-1], np.ones(n)))
    # Left boundary (x is 0, y varies)
    left_boundary = np.column_stack((np.zeros(n), boundary_spacing[::-1]))
    
    # Combine boundary points in sequence
    boundary_points = np.vstack([bottom_boundary, right_boundary, top_boundary, left_boundary])

    t_boundary = np.zeros((len(boundary_points), 1)) 
    for i in range(len(boundary_points)):
        if(boundary_points[i,1] == 0):
            t_boundary[i] = boundary_points[i,0]
        elif(boundary_points[i,0] == 1):
            t_boundary[i] = 1 + boundary_points[i,1]
        elif(boundary_points[i,1] == 1):
            t_boundary[i] = 3 - boundary_points[i,0]
        else:
            t_boundary[i] = 4 - boundary_points[i,1]
    
    return 1 / n, t_boundary, boundary_points

taus = [0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
h, s_for_integral, x_bd_g = generate_square_points(Ng)
s_for_intergal_pt = torch.from_numpy(s_for_integral).clone().float().to(device).view(-1, 1)
x_bd_g_pt = torch.from_numpy(x_bd_g).clone().float().to(device)


from scipy.special import kv, kvp

def kernel_int(tau, x, s):
    x1, x2 = x[0], x[1]
    if s < 1:
        x1_s, x2_s, x1_prime_s, x2_prime_s = s, 0, 1, 0
    elif s < 2:
        x1_s, x2_s, x1_prime_s, x2_prime_s = 1, s - 1, 0, 1
    elif s < 3:
        x1_s, x2_s, x1_prime_s, x2_prime_s = 3 - s, 1, - 1, 0
    else:
        x1_s, x2_s, x1_prime_s, x2_prime_s = 0, 4 - s, 0, -1
    r = np.sqrt((x1_s - x1) ** 2 + (x2_s - x2) ** 2)
    result = - kvp(0, r / np.sqrt(tau), n=1) * (x2_prime_s * (x1_s - x1) - x1_prime_s * (x2_s - x2)) / (np.sqrt(tau) * r)
    return result

x_int = x_int_pt.cpu().numpy()
kernels_for_int = torch.zeros(len(taus), len(x_int), len(s_for_integral))
for k in range(len(taus)):
    for i in range(len(x_int)):
        for j in range(len(s_for_integral)):
            kernels_for_int[k, i, j] = h * kernel_int(taus[k], x_int[i, :], s_for_integral[j, 0])


def predict_u_int(net, tau, g):      # tau: scalar, g: (batch_size, 4*n_g)
    flag = 0
    for i in range(len(taus)):
        if np.abs(tau - taus[i]) < 1e-6:
            kernel = kernels_for_int[i, :, :]
            flag = 1
            # print(taus[i])
    
    if flag == 0 and tau > 0:
        kernel = torch.zeros(len(x_int), len(s_for_integral))
        for i in range(len(x_int)):
            for j in range(len(s_for_integral)):
                kernel[i, j] = h * kernel_int(tau, x_int[i, :], s_for_integral[j, 0])
    else:
        exit("The value of tau is illegal!")

    kernel = kernel.to(device)

    result = 0.5 / (np.pi) * kernel @ net(tau, g).T
    return result.T

In [120]:
a = 0.5
b = np.sqrt(1 - a * a)

def U(x):
    x1 = x[:, 0].view(-1, 1)
    x2 = x[:, 1].view(-1, 1)
    return torch.sin(a * x1) * torch.cos(b * x2)

def F(x, tau):
    x1 = x[:, 0].view(-1, 1)
    x2 = x[:, 1].view(-1, 1)
    u = torch.sin(a * x1) * torch.cos(b * x2)
    laplace = - u
    return u - tau * laplace

def computeErrors(u_exact, u_pre, printOrNot):
    if isinstance(u_exact, np.ndarray):
        u_exact = torch.from_numpy(u_exact)
    if isinstance(u_pre, np.ndarray):
        u_pre = torch.from_numpy(u_pre)
    
    error = u_exact - u_pre
    l2_norm_abs = torch.norm(error, p=2).item() / torch.sqrt(torch.tensor(error.numel(), dtype=torch.float))
    max_norm_abs = torch.norm(error, p=float('inf')).item()
    l2_norm_rel = torch.norm(error, p=2).item() / torch.norm(u_exact, p=2).item()
    max_norm_rel = torch.norm(error, p=float('inf')).item() / torch.norm(u_exact, p=float('inf')).item()  
    
    l2_norm_rel_percent = l2_norm_rel * 100
    max_norm_rel_percent = max_norm_rel * 100
    
    if printOrNot == True:
        print(f"Absolute L2 Norm Error: {l2_norm_abs:.6f}")
        # print(f"Absolute Max Norm Error: {max_norm_abs:.6f}")
        # print(f"Relative L2 Norm Error: {l2_norm_rel_percent:.4f}%")
        # print(f"Relative Max Norm Error: {max_norm_rel_percent:.4f}%")

    return l2_norm_rel


def plotSolutions(u_exact, u_pre, N):
    if isinstance(u_exact, torch.Tensor):
        u_exact = u_exact.cpu().detach().numpy()
    if isinstance(u_pre, torch.Tensor):
        u_pre = u_pre.cpu().detach().numpy()

    u_exact = u_exact.reshape((N, N))
    u_pre = u_pre.reshape((N, N))
    error = u_exact - u_pre
    
    fig, axs = plt.subplots(1, 3, figsize=(20, 5))

    cax1 = axs[0].imshow(u_pre, cmap='viridis', extent=[0, 1, 0, 1], origin='lower')
    axs[0].set_title('Predicted Solution')
    fig.colorbar(cax1, ax=axs[0])

    cax2 = axs[1].imshow(u_exact, cmap='viridis', extent=[0, 1, 0, 1], origin='lower')
    axs[1].set_title('Exact Solution')
    fig.colorbar(cax2, ax=axs[1])

    cax3 = axs[2].imshow(np.abs(error), cmap='viridis', extent=[0, 1, 0, 1], origin='lower')
    axs[2].set_title('Error')
    fig.colorbar(cax3, ax=axs[2])

    plt.show()

In [121]:
def solveByNEKM(f, g, tau):      # f: (1, Nf*Nf), g: (1, 4*Ng), tau is a scalar
    tau_ = torch.zeros(f.shape[0], 1) + tau
    u1 = netF(f, x_pt, tau_.to(device))
    u2 = predict_u_int(netg, tau, g)

    u_pred_f = torch.zeros_like(f).to(device)
    u_pred_g = torch.zeros_like(f).to(device)
    u_pred_f[:, ~boundary_mask] = u1[:, ~boundary_mask]
    u_pred_g[:, ~boundary_mask] = u2
    u_pred_g[:, boundary_mask] = U(x_bd_pt).view(1, -1)

    return u_pred_f, u_pred_g

In [122]:
def evaluate(modelD_g, modelD_f, tau):
    x_vals = np.linspace(0, 1, 41)
    X, Y = np.meshgrid(x_vals, x_vals)
    x_grid = np.stack([X.flatten(), Y.flatten()], axis=1)  # (1681, 2)
    x_tensor = torch.tensor(x_grid, dtype=torch.float32).to(device)

    with torch.no_grad():
        f_tensor = F(x_tensor, tau).reshape(1, -1)
        u_exact_tensor = U(x_tensor).reshape(1, -1)       # (1, 1681)

    u_np_2d = u_exact_tensor.cpu().numpy().reshape(41, 41)
    g_boundary = np.concatenate([u_np_2d[0, :], u_np_2d[1:, 40], u_np_2d[40, 39::-1], u_np_2d[39:0:-1, 0]])

    g_tensor = torch.tensor(g_boundary.reshape(1, -1), dtype=torch.float32).to(device)  # (1, 160)
    k_tensor = torch.tensor([[tau]], dtype=torch.float32).to(device)

    modelD_g.eval()
    with torch.no_grad():
        u_pred_tensor_g_DeepONet = modelD_g(k_tensor, g_tensor, x_tensor)  # (1, 1681)

    modelD_f.eval()
    with torch.no_grad():
        u_pred_tensor_f_DeepONet = modelD_f(k_tensor, f_tensor, x_tensor)  # (1, 1681)

    u_pred_tensor_f_NEKM, u_pred_tensor_g_NEKM = solveByNEKM(f_tensor, U(x_bd_g_pt).view(1, -1), tau)

    # print(u_pred_tensor_g_DeepONet.shape, u_pred_tensor_f_DeepONet.shape, u_pred_tensor_g_NEKM.shape, u_pred_tensor_f_NEKM.shape)

    u_exact = u_exact_tensor.view(-1)
    u_pred1 = (u_pred_tensor_g_DeepONet + u_pred_tensor_f_DeepONet).view(-1)
    u_pred2 = (u_pred_tensor_g_DeepONet + u_pred_tensor_f_NEKM).view(-1)
    u_pred3 = (u_pred_tensor_g_NEKM + u_pred_tensor_f_DeepONet).view(-1)
    u_pred4 = (u_pred_tensor_g_NEKM + u_pred_tensor_f_NEKM).view(-1)

    # plotSolutions(u_exact, u_pred1, N=41)
    computeErrors(u_exact, u_pred1, 1)

    # plotSolutions(u_exact, u_pred2, N=41)
    computeErrors(u_exact, u_pred2, 1)

    # plotSolutions(u_exact, u_pred3, N=41)
    computeErrors(u_exact, u_pred3, 1)

    # plotSolutions(u_exact, u_pred4, N=41)
    computeErrors(u_exact, u_pred4, 1)

In [123]:
evaluate(net_g, net_f, 0.05)

Absolute L2 Norm Error: 0.005124
Absolute L2 Norm Error: 0.002957
Absolute L2 Norm Error: 0.004095
Absolute L2 Norm Error: 0.000222


In [124]:
evaluate(net_g, net_f, 0.06)

Absolute L2 Norm Error: 0.005869
Absolute L2 Norm Error: 0.002867
Absolute L2 Norm Error: 0.003745
Absolute L2 Norm Error: 0.000141


In [125]:
evaluate(net_g, net_f, 0.07)

Absolute L2 Norm Error: 0.006572
Absolute L2 Norm Error: 0.003188
Absolute L2 Norm Error: 0.003734
Absolute L2 Norm Error: 0.000148


In [126]:
evaluate(net_g, net_f, 0.08)

Absolute L2 Norm Error: 0.006146
Absolute L2 Norm Error: 0.003209
Absolute L2 Norm Error: 0.003490
Absolute L2 Norm Error: 0.000125


In [127]:
evaluate(net_g, net_f, 0.09)

Absolute L2 Norm Error: 0.005474
Absolute L2 Norm Error: 0.002954
Absolute L2 Norm Error: 0.003588
Absolute L2 Norm Error: 0.000100


In [128]:
evaluate(net_g, net_f, 0.1)

Absolute L2 Norm Error: 0.004745
Absolute L2 Norm Error: 0.003624
Absolute L2 Norm Error: 0.003601
Absolute L2 Norm Error: 0.000154


In [129]:
class ModelD(nn.Module):
    def __init__(self, f_dim, g_dim, hidden_dim):
        super().__init__()
        # Branch Net dim_input = 1 + f_dim + g_dim
        self.f_dim = f_dim
        self.g_dim = g_dim
        self.branch_net = nn.Sequential(
            nn.Linear(1 + f_dim + g_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        self.trunk_net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, k, f, g, x):
        """
        k: (B, 1)
        f: (B, f_dim)
        g: (B, g_dim)
        x: (M, 2)   
        """
        branch_input = torch.cat([k, f, g], dim=1)  # shape (B, 1+f_dim+g_dim)
        branch_out = self.branch_net(branch_input)  # shape (B, H)
        trunk_out = self.trunk_net(x)               # shape (M, 2)

        return torch.einsum('bi,mi->bm', branch_out, trunk_out)

In [130]:
def evaluateModelD(model, tau):
    x_vals = np.linspace(0, 1, 41)
    X, Y = np.meshgrid(x_vals, x_vals)
    x_grid = np.stack([X.flatten(), Y.flatten()], axis=1)  # (1681, 2)
    x_tensor = torch.tensor(x_grid, dtype=torch.float32).to(device)

    with torch.no_grad():
        f_tensor = F(x_tensor, tau).reshape(1, -1)        # (1, 1681)
        u_exact_tensor = U(x_tensor).reshape(1, -1)       # (1, 1681)

    # Build g from f (only use boundary entries)
    u_np_2d = u_exact_tensor.cpu().numpy().reshape(41, 41)
    g_boundary = np.concatenate([
        u_np_2d[0, :],            
        u_np_2d[1:, 40],          
        u_np_2d[40, 39::-1],        
        u_np_2d[39:0:-1, 0],       
    ])

    g_tensor = torch.tensor(g_boundary.reshape(1, -1), dtype=torch.float32).to(device)  # (1, 160)
    k_tensor = torch.tensor([[tau]], dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        u_pred_tensor = model(k_tensor, f_tensor, g_tensor, x_tensor)  # (1, 1681)

    u_exact = u_exact_tensor.view(-1)
    u_pred = u_pred_tensor.view(-1)

    # plotSolutions(u_exact, u_pred, N=41)
    computeErrors(u_exact, u_pred, 1)

In [131]:
netD = ModelD(f_dim = M, g_dim = 4 * NN, hidden_dim = 3 * M // 4).to(device)
state_dict = torch.load("./DeepONet/net.pth", map_location=device, weights_only=True)
netD.load_state_dict(state_dict)
netD.eval()  

ModelD(
  (branch_net): Sequential(
    (0): Linear(in_features=1842, out_features=1260, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1260, out_features=1260, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1260, out_features=1260, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1260, out_features=1260, bias=True)
    (7): ReLU()
    (8): Linear(in_features=1260, out_features=1260, bias=True)
  )
  (trunk_net): Sequential(
    (0): Linear(in_features=2, out_features=1260, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1260, out_features=1260, bias=True)
    (3): ReLU()
    (4): Linear(in_features=1260, out_features=1260, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1260, out_features=1260, bias=True)
    (7): ReLU()
    (8): Linear(in_features=1260, out_features=1260, bias=True)
  )
)

In [132]:
evaluateModelD(netD, 0.05)
evaluateModelD(netD, 0.06)
evaluateModelD(netD, 0.07)
evaluateModelD(netD, 0.08)
evaluateModelD(netD, 0.09)
evaluateModelD(netD, 0.1)

Absolute L2 Norm Error: 0.047516
Absolute L2 Norm Error: 0.046682
Absolute L2 Norm Error: 0.045906
Absolute L2 Norm Error: 0.045201
Absolute L2 Norm Error: 0.044508
Absolute L2 Norm Error: 0.043822
